# Session 4 · Classification and Decision Trees

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/01-data-mining-with-weka/code/session-04-classification-and-decision-trees.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

### Example 1 · Entropy and information gain from scratch

In [1]:
from math import log2

def entropy(counts):
    total = sum(counts)
    return -sum(c / total * log2(c / total) for c in counts if c)

parent = entropy([6, 4])
high, low = entropy([5, 1]), entropy([1, 3])
weighted = 6 / 10 * high + 4 / 10 * low
print(f"H(parent) = {parent:.3f}")
print(f"H(high)   = {high:.3f}   H(low) = {low:.3f}")
print(f"Weighted  = {weighted:.3f}")
print(f"Gain      = {parent - weighted:.3f}")

H(parent) = 0.971
H(high)   = 0.650   H(low) = 0.811
Weighted  = 0.715
Gain      = 0.256


### Example 5 · The same tree in scikit-learn

In [2]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import cross_val_score

rows = """sunny,hot,high,FALSE,no
sunny,hot,high,TRUE,no
overcast,hot,high,FALSE,yes
rainy,mild,high,FALSE,yes
rainy,cool,normal,FALSE,yes
rainy,cool,normal,TRUE,no
overcast,cool,normal,TRUE,yes
sunny,mild,high,FALSE,no
sunny,cool,normal,FALSE,yes
rainy,mild,normal,FALSE,yes
sunny,mild,normal,TRUE,yes
overcast,mild,high,TRUE,yes
overcast,hot,normal,FALSE,yes
rainy,mild,high,TRUE,no"""
df = pd.DataFrame([r.split(",") for r in rows.split("\n")],
                  columns=["outlook", "temperature", "humidity", "windy", "play"])
X = pd.get_dummies(df.drop(columns="play")).astype(int)
y = df["play"]

tree = DecisionTreeClassifier(criterion="entropy", random_state=0).fit(X, y)
print(export_text(tree, feature_names=list(X.columns)))
scores = cross_val_score(DecisionTreeClassifier(criterion="entropy", random_state=0), X, y, cv=7)
print("7-fold CV accuracy:", scores.round(2), "mean =", round(scores.mean(), 3))

|--- outlook_overcast <= 0.50
|   |--- humidity_normal <= 0.50
|   |   |--- outlook_rainy <= 0.50
|   |   |   |--- class: no
|   |   |--- outlook_rainy >  0.50
|   |   |   |--- windy_FALSE <= 0.50
|   |   |   |   |--- class: no
|   |   |   |--- windy_FALSE >  0.50
|   |   |   |   |--- class: yes
|   |--- humidity_normal >  0.50
|   |   |--- windy_TRUE <= 0.50
|   |   |   |--- class: yes
|   |   |--- windy_TRUE >  0.50
|   |   |   |--- temperature_cool <= 0.50
|   |   |   |   |--- class: yes
|   |   |   |--- temperature_cool >  0.50
|   |   |   |   |--- class: no
|--- outlook_overcast >  0.50
|   |--- class: yes

7-fold CV accuracy: [1.  0.5 0.5 0.5 0.5 0.5 0.5] mean = 0.571


### Example 6 · Computing the metrics yourself

In [3]:
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score

actual    = ["yes"] * 9 + ["no"] * 5
predicted = ["yes", "yes", "yes", "yes", "yes", "yes", "yes", "no", "no",
             "yes", "no", "no", "no", "yes"]
print("Confusion matrix (rows = actual yes/no):")
print(confusion_matrix(actual, predicted, labels=["yes", "no"]))
print()
print(classification_report(actual, predicted, labels=["yes", "no"], digits=3))
print("Kappa:", round(cohen_kappa_score(actual, predicted), 3))

Confusion matrix (rows = actual yes/no):
[[7 2]
 [2 3]]

              precision    recall  f1-score   support

         yes      0.778     0.778     0.778         9
          no      0.600     0.600     0.600         5

    accuracy                          0.714        14
   macro avg      0.689     0.689     0.689        14
weighted avg      0.714     0.714     0.714        14

Kappa: 0.378
